# Cadastral Pasture Parcel Segmentation - LAPIG / UFG

Advanced modular pipeline for delimiting pasture subdivisions (parcels) and cadastral management units:
- **Multi-Temporal Composites:** Sentinel-2 Harmonized CS+ (2025)
- **Segment Anything AI:** SAMGeo (Segment Anything Model - ViT-H / ViT-L / ViT-B)
- **Cadastral Regularization:** Planar graph, post-to-post straight fence line chord rectification, and enclave absorption
- **Pure Segmentation (Unfiltered):** Full physical extraction of **all agricultural parcels** within the CAR property perimeter
- **Zonal MapBiomas Filter:** Thematic classification with MapBiomas 10m (Col 4) / 30m (Col 11) up to 2025
- **Automated Hardware Benchmark:** Real-time background tracking of VRAM, RAM, CPU %, GPU Load %, thermals, and runtime per phase with an auditable technical report.

### 1. Environment & Earth Engine Initialization
> If your Google Cloud account requires a specific project ID, provide it in `ee.Initialize(project='your-project')`.
> The pipeline automatically detects the active Earth Engine session.

In [1]:
import os
import sys
import ee
import geopandas as gpd

ee_project = 'mapbiomas-brazil'

# Add repository directory to path for local module import
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Initialize Earth Engine
try:
    ee.Initialize(project=ee_project)
    print('Earth Engine initialized successfully!')
except Exception as e:
    # Example: ee.Initialize(project='ee-vieiramesquita')
    print(f'Notice on automatic initialization: {e}')
    print('Tip: If required, specify project: ee.Initialize(project="...")')

# Import core functions from LAPIG package
from lapig_parcel_segmentation import (
    run_analysis_lapig,
    run_batch_analysis_lapig,
    resolve_input_roi,
    HardwareBenchmarkMonitor
)
print('LAPIG parcel segmentation package loaded successfully!')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


✅ Earth Engine inicializado com sucesso!
✅ Pacote LAPIG carregado com sucesso!


### 2. Inspect the CAR Vector File (Goiás)
Loads the GeoPackage containing rural properties and previews the first property IDs (`cod_imovel`).

In [2]:
input_gpkg = "car_goias_area_imovel_epsg4674_hottest.gpkg"
id_col = "cod_imovel"

gdf_base = gpd.read_file(input_gpkg)
print(f"Total properties found: {len(gdf_base)}")

# Display key attributes
cols = [c for c in [id_col, 'municipio', 'num_area'] if c in gdf_base.columns]
gdf_base[cols].head(10)

📋 Total de imóveis encontrados: 19


,cod_imovel,municipio,num_area
0,GO-5203500-366A293867A24EF5A446DA91E23F8548,Bom Jesus de Goias,4476.3120
1,GO-5203807-146172AFEDAF44FC82CEEC3A2B92D5F1,Britania,2629.7952
2,GO-5204409-01D1AD1F9586415E9D46BF93115D27DF,Caiaponia,798.2250
3,GO-5205521-DA104FC8E15A44DFB6CB4F0656C1E409,Colinas do Sul,896.5830
4,GO-5206404-499D0D75A1394A6D915AE91FF8DB8700,Crixas,233.8072
5,GO-5207402-1D40C8AAD6504E05B30DA04D7E19457F,Edeia,530.0171
6,GO-5208707-4045057E90C948D999280F125FD7FD6B,Goiania,70.2766
7,GO-5208905-F496428818874B9CBBA57AD8132348B1,Goias,10828.8002
8,GO-5208905-08E150868E2A43CDBB23AC8FEE459017,Goias,828.9465
9,GO-5209903-C8653C1E63894890B5883B8529A3A100,Iaciara,532.2021


### 3. Option A: Run for a Specific Property (With Pure Segmentation & Benchmark)

During execution, the pipeline:
1. **Monitors Hardware:** Continuously tracks CPU, RAM, VRAM, and GPU compute in real time.
2. **Generates Pure Segmentation (`parcelas_segmentacao_pura_{ID}.shp`):** All physical parcels inside the CAR boundary, annotated with audit attributes (`pasto_cov`, `ndvi_med`, `is_pasto`, `classe`).
3. **Generates Pasture Parcels (`parcelas_finais_cercas_retas_{ID}.shp`):** Parcels validated via MapBiomas zonal dominance.
4. **Generates Technical Benchmark Report:** `relatorio_benchmark_hardware.txt` and `.json` in the property folder.

In [15]:
# Select property as example
cod_alvo = gdf_base[id_col].iloc[0]
print(f"Target property selected: {cod_alvo}")

# Execute analysis returning both pasture layer and pure physical segmentation
gdf_pasto, shp_pasto, gdf_puro, shp_puro = run_analysis_lapig(
    input_target=input_gpkg,
    id_column=id_col,
    id_value=cod_alvo,
    year=2025,                  # Target year (Sentinel-2 and MapBiomas 2025)
    mapbiomas_collection='10m', # MapBiomas Collection 4 (10m) up to 2025
    min_area_ha=2.0,            # Filters out slivers under 2 ha
    straight_tol=22.0,          # Fence chord rectification tolerance (meters)
    min_pasture_coverage=0.35,  # Zonal pasture dominance threshold (>= 35%)
    skip_sam_if_exists=True,    # Reuses masks.tif if previously generated
    return_pure=True,           # Returns GeoDataFrame and path for pure segmentation
    enable_benchmark=True,      # Activates continuous hardware benchmark tracking
    ee_project=ee_project
)

print(f"\nPasture Shapefile:        {shp_pasto} ({len(gdf_pasto)} parcels)")
print(f"Pure Physical Shapefile:  {shp_puro} ({len(gdf_puro)} parcels)")

IndexError: single positional indexer is out-of-bounds

### 4. Technical Hardware Benchmark Report
Displays consolidated hardware resource consumption metrics recorded during property execution:

In [16]:
report_txt = os.path.join(os.path.dirname(shp_pasto), "relatorio_benchmark_hardware.txt")
if os.path.exists(report_txt):
    with open(report_txt, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("Benchmark report not found.")

📊 RELATÓRIO DE BENCHMARK DE HARDWARE - LAPIG CADASTRAL PIPELINE
⏱️  TEMPO TOTAL DE PROCESSAMENTO: 0:07:40 (460.9s)
🖥️  DISPOSITIVO DE PROCESSAMENTO: NVIDIA GeForce RTX 2080 Ti
--------------------------------------------------------------------------------
⏱️  DISTRIBUIÇÃO DE TEMPO POR ETAPA:
  • Inicialização                              0:00:00 (  0.0%)
  • 1. Resolução da Área de Interesse (ROI)    0:00:01 (  0.3%)
  • 2. Download Mosaico Sentinel-2 (GEE)       0:00:59 ( 12.9%)
  • 3. Inferência SAMGeo (ViT-H GPU)           0:06:32 ( 85.2%)
  • 4. Vetorização e Regularização Cadastral (CPU) 0:00:01 (  0.3%)
  • 5. Segmentação Pura no Perímetro CAR       0:00:00 (  0.0%)
  • 6. Classificação Zonal MapBiomas (GEE/CPU) 0:00:06 (  1.4%)
  • 7. Exportação e Estatísticas Cadastrais    0:00:00 (  0.0%)
--------------------------------------------------------------------------------
🎮 PLACA DE VÍDEO (GPU - NVIDIA GeForce RTX 2080 Ti):
  • Pico de VRAM Alocada:           6.47 GB / 11.00 GB (

### 5. Option B: Batch Processing for Multiple Properties
> Iterates sequentially over properties in the vector dataset.
> Creates individual `results/{cod_imovel}/` subfolders, consolidated shapefiles, and batch benchmark report.

In [3]:
import os
import sys
import ee
import geopandas as gpd

input_gpkg = "car_goias_area_imovel_epsg4674_hottest.gpkg"
id_col = "cod_imovel"

gdf_base = gpd.read_file(input_gpkg)
print(f"Total properties found: {len(gdf_base)}")

# Display key attributes
cols = [c for c in [id_col, 'municipio', 'num_area'] if c in gdf_base.columns]
gdf_base[cols].head(10)

# Select first property as test sample
cod_alvo = gdf_base[id_col].iloc[0]
print(f"Target property selected: {cod_alvo}")

📋 Total de imóveis encontrados: 19
🎯 Imóvel selecionado: GO-5207402-1D40C8AAD6504E05B30DA04D7E19457F


##### Set max_imoveis=3 for a quick batch test or max_imoveis=None for the entire dataset
```python
gdfs_sucesso, shp_consolidado = run_batch_analysis_lapig(
    input_target=input_gpkg,
    id_column=id_col,
    max_imoveis=2,              # None = all properties in file
    year=2025,
    mapbiomas_collection='10m',
    min_area_ha=2.0,
    straight_tol=22.0,
    min_pasture_coverage=0.35,
    skip_sam_if_exists=True,
    consolidate_shp=True,
    enable_benchmark=True,
    ee_project=ee_project
)

print(f"\nBatch execution complete!")
print(f"Consolidated Pasture Shapefile: {shp_consolidado}")
print(f"Consolidated Pure Shapefile:     results/parcelas_todos_imoveis_segmentacao_pura.shp")
```

### 6. Interactive Visualization in JupyterLab (Leaflet / Folium)
Compare the **Pure Physical Segmentation** (all agricultural parcels) and the **Pasture Parcels** validated by MapBiomas:

In [ ]:
if 'gdf_puro' in locals() and len(gdf_puro) > 0:
    # Visualize pure physical parcels colored by classification (Pasture vs Others)
    display(gdf_puro.explore(
        column='classe',
        cmap='tab10',
        tooltip=['id_parcela', 'area_ha', 'classe', 'pasto_cov', 'ndvi_med'],
        popup=True,
        tiles='CartoDB positron',
        legend=True
    ))
elif 'gdf_pasto' in locals() and len(gdf_pasto) > 0:
    display(gdf_pasto.explore(
        column='area_ha',
        cmap='YlGn',
        tooltip=['id_parcela', 'area_ha', 'pasto_cov'],
        popup=True,
        tiles='CartoDB positron'
    ))

In [5]:
# Example of running a specific property from Parquet
# gdfs_sucesso, shp_consolidado = run_batch_analysis_lapig(
#     input_target="car_goias_area_imovel_epsg4674_hottest.gpkg",
#     id_value='GO-5211008-46C08C6235D74D36BE07B5384A17B156',
#     year=2025,
#     mapbiomas_collection='10m',
#     min_area_ha=2.0,
#     straight_tol=22.0,
#     min_pasture_coverage=0.35,
#     skip_sam_if_exists=True,
#     consolidate_shp=True,
#     enable_benchmark=True,
#     ee_project=ee_project
# )



🚜 INICIANDO PROCESSAMENTO EM LOTE (BATCH) - LAPIG
   Arquivo de Entrada: CAR_BRASIL_AREA_IMOVEL_EPSG4674.parquet
   Coluna de ID:       cod_imovel
📋 Total de imóveis a serem processados: 1

################################################################################
🚜 [1/1] PROCESSANDO IMÓVEL: GO-5211008-46C08C6235D74D36BE07B5384A17B156
################################################################################
🧹 [VRAM Clean] Memória GPU liberada: Alocada=9.1 MB | Reservada=14.0 MB
📥 Entrada identificada: Arquivo Vetorial Local (.parquet) -> CAR_BRASIL_AREA_IMOVEL_EPSG4674.parquet
🎯 Imóvel filtrado pela coluna 'cod_imovel' == 'GO-5211008-46C08C6235D74D36BE07B5384A17B156'

🚀 PIPELINE DE SEGMENTAÇÃO CADASTRAL LAPIG
   Imóvel / ID:      GO-5211008-46C08C6235D74D36BE07B5384A17B156
   Pasta de Saída:   results/GO-5211008-46C08C6235D74D36BE07B5384A17B156
   Ano Base:         2025 (Janela seca: meses 03 a 10)
   MapBiomas:        Coleção 10m (cobertura até 2025)
🛰️ Exportando mosaic